# 少量示例的提示词模版的使用

在构建prompt时，可以通过构建一个 少量示例列表 去进一步格式化prompt，这是一种简单但强大的指
导生成的方式，在某些情况下可以 显著提高模型性能 。
少量示例提示模板可以由 一组示例 或一个负责从定义的集合中选择 一部分示例 的示例选择器构建。
前者：使用 FewShotPromptTemplate 或 FewShotChatMessagePromptTemplate
后者：使用 Example selectors(示例选择器)
每个示例的结构都是一个 字典 ，其中 键 是输入变量， 值 是输入变量的值。

FewShotPromptTemplate :与PromptTemplate一起使用

FewShotChatMessagePromptTemplate：与ChatPromptTemplate一起使用

Example selectors(示例选择器)：

## 1.FewShotPromptTemplate的使用

示例1：未提供示例的情况

In [1]:
import os
import dotenv
from langchain_openai import ChatOpenAI

# 1.提供大模型
# 加载配置文件
dotenv.load_dotenv()
# 设置环境
os.environ['OPENAI_API_KEY'] = os.getenv("OPENAI_API_KEY1")
os.environ['OPENAI_BASE_URL'] = os.getenv("OPENAI_BASE_URL")
# 获取对话模型：
chat_model = ChatOpenAI(model="gpt-4o-mini",
                        temperature=0.4)
res = chat_model.invoke("2 🦜 9是多少?")
print(res.content)
'''
2 🦜 9 的意思可能是将数字 2 和 9 结合在一起，但具体的运算符号不明确。如果你是想进行加法，那么 2 + 9 = 11。如果有其他的意思，请提供更多的上下文！
'''


2 🦜 9 的意思可能是将数字 2 和 9 结合在一起，但具体的运算符号不明确。如果你是想进行加法，那么 2 + 9 = 11。如果有其他的意思，请提供更多的上下文！


举例2:使用FewShotPromptTemplate

In [13]:
from langchain_core.prompts import PromptTemplate, FewShotPromptTemplate

#1 创建示例集合
examples = [
    {"input": "北京天气怎么样", "output": "北京市"},
    {"input": "南京下雨吗", "output": "南京市"},
    {"input": "武汉热吗", "output": "武汉市"}
]
#2 创建PromptTemplate实例
example_prompt = PromptTemplate.from_template(
    template="Input:{input}\nOutput:{output}"
)
#3 创建FewShotPromptTemplate实例
prompt = FewShotPromptTemplate(
    examples=examples,
    example_prompt=example_prompt,
    suffix="Input:{input}\nOutput:",
    input_variables=["input"]
)

# 4调用
# prompt = prompt.invoke({"input":"长沙多少度"})
# print("===Prompt===")
# print(prompt)


'''
===Prompt===
text='Input:北京天气怎么样\nOutput:北京市\n\nInput:南京下雨吗\nOutput:南京市\n\nInput:武汉热吗\nOutput:武汉市\n\nInput:长沙多少度\nOutput:'
'''


"\n===Prompt===\ntext='Input:北京天气怎么样\nOutput:北京市\n\nInput:南京下雨吗\nOutput:南京市\n\nInput:武汉热吗\nOutput:武汉市\n\nInput:长沙多少度\nOutput:'\n"

In [7]:
import os
import dotenv
from langchain_openai import ChatOpenAI

# 1.提供大模型
# 加载配置文件
dotenv.load_dotenv()
# 设置环境
os.environ['OPENAI_API_KEY'] = os.getenv("OPENAI_API_KEY1")
os.environ['OPENAI_BASE_URL'] = os.getenv("OPENAI_BASE_URL")
# 获取对话模型：
chat_model = ChatOpenAI(model="gpt-4o-mini")

In [14]:
response = chat_model.invoke(prompt.invoke({"input": "长沙多少度"}))  # 这边注意输入的类型

print(response.content)

长沙市


## 2 FewShotChatMessagePromptTemplate：与ChatPromptTemplate一起使用

In [16]:
from langchain_core.prompts import (FewShotChatMessagePromptTemplate, ChatPromptTemplate)

# 1.示例消息格式
examples = [
    {"input": "1+1等于几？", "output": "1+1等于2"},
    {"input": "法国的首都是？", "output": "巴黎"}
]
# 2.定义示例的消息格式提示词模版
msg_example_prompt = ChatPromptTemplate.from_messages([
    ("human", "{input}"),
    ("ai", "{output}"),
])
# 3.定义FewShotChatMessagePromptTemplate对象
few_shot_prompt = FewShotChatMessagePromptTemplate(
    example_prompt=msg_example_prompt,
    examples=examples
)
# 4.输出格式化后的消息
print(few_shot_prompt.format())

'''
Human: 1+1等于几？
AI: 1+1等于2
Human: 法国的首都是？
AI: 巴黎
'''

Human: 1+1等于几？
AI: 1+1等于2
Human: 法国的首都是？
AI: 巴黎


In [5]:
# 1.导入相关包
from langchain_core.prompts import (FewShotChatMessagePromptTemplate,
                                    ChatPromptTemplate)

# 2.定义示例组
examples = [
    {"input": "2🦜2", "output": "4"},
    {"input": "2🦜3", "output": "8"},
]
# 3.定义示例的消息格式提示词模版
example_prompt = ChatPromptTemplate.from_messages([
    ('human', '{input} 是多少?'),
    ('ai', '{output}')
])
# 4.定义FewShotChatMessagePromptTemplate对象
few_shot_prompt = FewShotChatMessagePromptTemplate(
    examples=examples,
    example_prompt=example_prompt
)
# 5.输出完整提示词的消息模版
final_prompt = ChatPromptTemplate.from_messages(
    [
        ('system', '你是一个数学奇才'),
        few_shot_prompt,
        ('human', '{input}'),
    ]
)
#6.提供大模型
import os
import dotenv
from langchain_openai import ChatOpenAI

dotenv.load_dotenv()
os.environ['OPENAI_API_KEY'] = os.getenv("OPENAI_API_KEY1")
os.environ['OPENAI_BASE_URL'] = os.getenv("OPENAI_BASE_URL")
chat_model = ChatOpenAI(model="gpt-4o-mini",
                        temperature=0.4)
chat_model.invoke(final_prompt.invoke(input="2🦜4")).content



'16'

## 3.Example selectors(示例选择器)
使用的好处：避免盲目传递所有示例，减少 token 消耗的同时，还可以提升输出效果。


示例选择策略：语义相似选择、长度选择、最大边际相关示例选择等


语义相似选择 ：通过余弦相似度等度量方式评估语义相关性，选择与输入问题最相似的 k 个示
例。
长度选择 ：根据输入文本的长度，从候选示例中筛选出长度最匹配的示例。增强模型对文本结构的
理解。比语义相似度计算更轻量，适合对响应速度要求高的场景。
最大边际相关示例选择 ：优先选择与输入问题语义相似的示例；同时，通过惩罚机制避免返回同质
化的内容
余弦相似度是通过计算两个向量的夹⻆余弦值来衡量它们的相似性。它的值范围在-1到1之
间：当两个向量⽅向相同时值为1；夹⻆为90°时值为0；⽅向完全相反时为-1。
数学表达式：余弦相似度 = (A·B) / (||A|| * ||B||)。其中A·B是点积，||A||和||B||是向量的模（⻓
度）

In [ ]:
# 1.导入相关包
from langchain_community.vectorstores import Chroma
from langchain_core.example_selectors import SemanticSimilarityExampleSelector
import os
import dotenv
from langchain_openai import OpenAIEmbeddings

dotenv.load_dotenv()

# 2.定义嵌入模型
os.environ['OPENAI_API_KEY'] = os.getenv("OPENAI_API_KEY1")

os.environ['OPENAI_BASE_URL'] = os.getenv("OPENAI_BASE_URL")
embeddings_model = OpenAIEmbeddings(
    model="text-embedding-ada-002"
)
# 3.定义示例组
examples = [
    {
        "question": "谁活得更久，穆罕默德·阿里还是艾伦·图灵?",
        "answer": """
接下来还需要问什么问题吗？
追问：穆罕默德·阿里去世时多大年纪？
中间答案：穆罕默德·阿里去世时享年74岁。
""",
    },
    {
        "question": "craigslist的创始人是什么时候出生的？",
        "answer": """
接下来还需要问什么问题吗？
追问：谁是craigslist的创始人？
中级答案：Craigslist是由克雷格·纽马克创立的。
""",
    },
    {
        "question": "谁是乔治·华盛顿的外祖父？",
        "answer": """
接下来还需要问什么问题吗？
追问：谁是乔治·华盛顿的母亲？
中间答案：乔治·华盛顿的母亲是玛丽·鲍尔·华盛顿。
""",
    },
    {
        "question": "《大白鲨》和《皇家赌场》的导演都来自同一个国家吗？",
        "answer": """
接下来还需要问什么问题吗？
追问：《大白鲨》的导演是谁？
中级答案：《大白鲨》的导演是史蒂文·斯皮尔伯格。
""",
    },
]
# 4.定义示例选择器
example_selector = SemanticSimilarityExampleSelector.from_examples(
    # 这是可供选择的示例列表
    examples,
    # 这是用于生成嵌入的嵌入类，用于衡量语义相似性
    embeddings_model,
    # 这是用于存储嵌入并进行相似性搜索的 VectorStore 类
    Chroma,
    # 这是要生成的示例数量
    k=1,
)
# 选择与输入最相似的示例
question = "玛丽·鲍尔·华盛顿的父亲是谁?"
selected_examples = example_selector.select_examples({"question": question})
print(f"与输入最相似的示例：{selected_examples}")





In [3]:
# 1.导入相关包
from langchain_community.vectorstores import FAISS
from langchain_core.example_selectors import SemanticSimilarityExampleSelector
from langchain_core.prompts import FewShotPromptTemplate, PromptTemplate
from langchain_openai import OpenAIEmbeddings

# 2.定义示例提示词模版
example_prompt = PromptTemplate.from_template(
    template="Input: {input}\nOutput: {output}",
)
# 3.创建一个示例提示词模版
examples = [
    {"input": "高兴", "output": "悲伤"},
    {"input": "高", "output": "矮"},
    {"input": "长", "output": "短"},
    {"input": "精力充沛", "output": "无精打采"},
    {"input": "阳光", "output": "阴暗"},
    {"input": "粗糙", "output": "光滑"},
    {"input": "干燥", "output": "潮湿"},
    {"input": "富裕", "output": "贫穷"},
]
# 4.定义嵌入模型
embeddings = OpenAIEmbeddings(
    model="text-embedding-ada-002"
)
# 5.创建语义相似性示例选择器
example_selector = SemanticSimilarityExampleSelector.from_examples(
    examples,
    embeddings,
    FAISS,
    k=2,
)
#或者
#example_selector = SemanticSimilarityExampleSelector(
# examples,
# embeddings,
# FAISS,
# k=2
#)
# 6.定义小样本提示词模版
similar_prompt = FewShotPromptTemplate(
    example_selector=example_selector,
    example_prompt=example_prompt,
    prefix="给出每个词组的反义词",
    suffix="Input: {word}\nOutput:",
    input_variables=["word"],
)
response = similar_prompt.invoke({"word": "忧郁"})
print(response.text)

'''
给出每个词组的反义词

Input: 高兴
Output: 悲伤

Input: 阳光
Output: 阴暗

Input: 忧郁
Output:
'''


给出每个词组的反义词

Input: 高兴
Output: 悲伤

Input: 阳光
Output: 阴暗

Input: 忧郁
Output:
